In [ ]:
#import the libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import matplotlib.pyplot as plt
import seaborn as sns

import joblib

In [ ]:
#Load preprocessed data
df = pd.read_csv('../data/processed/flight_delays_preprocessed.csv')
df.head()

In [ ]:
#Define features and target variable
TARGET = 'DelayMinutes'

x = df.drop(columns=[TARGET])
y = df[TARGET]

In [ ]:
#Train?Test Split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
#Train RandomForest Regressor
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(x_train, y_train)


In [ ]:
#prediction
y_pred = rf.predict(x_test)

In [ ]:
#Model Evaluation
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred)) 

print(f"MAE  : {mae:.2f}")
print(f"MSE  : {mse:.2f}")
print(f"RMSE : {rmse:.2f}")


In [ ]:
#Hyperparameter Tuning
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

grid_search.fit(x_train, y_train)

print("Best Parameters:", grid_search.best_params_)


In [ ]:
#Model training with best parameters
final_model = grid_search.best_estimator_
final_model.fit(x_train, y_train)

final_pred = final_model.predict(x_test)

final_mae = mean_absolute_error(y_test, final_pred)
final_mse = mean_squared_error(y_test, final_pred)
final_rmse = np.sqrt(mean_squared_error(y_test, final_pred))


print(f"Final MAE  : {final_mae:.2f}")
print(f"Final MSE  : {final_mse:.2f}")
print(f"Final RMSE : {final_rmse:.2f}")

In [ ]:
#Feature Importance
feature_importances = pd.Series(final_model.feature_importances_, index=x.columns)
feature_importances = feature_importances.sort_values(ascending=False)

feature_importances.head(10)

In [ ]:
plt.figure(figsize=(8,5))
feature_importances.head(10).plot(kind="bar")
plt.title("Top 10 Important Features for Flight Delay")
plt.ylabel("Importance")
plt.show()


In [ ]:
#save the model
joblib.dump(final_model, "../models/flight_delay_model.pkl")


In [ ]:
#Load and test the saved model
loaded_model = joblib.load("../models/flight_delay_model.pkl")

loaded_model.predict(x_test.iloc[:5])
